# Shown Space Scoring Path Visuals

This notebook uses Shown Space's public game API to recreate field-path visuals from coordinates, then summarizes a team's scoring possessions as an interactive average path and heatmap.

Default team: `glory`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd

from ufa import (
    average_scoring_path,
    build_scoring_possessions,
    fetch_shownspace_games,
    fetch_shownspace_season_throws,
    plot_average_scoring_path,
    plot_scoring_heatmap,
)

## Settings

Use `MAX_GAMES = 3` first to validate the visual quickly. Set `MAX_GAMES = None` for the full team season.

In [ ]:
SEASON = 2026
TEAM_ID = "glory"
MAX_GAMES = 3

games, throws = fetch_shownspace_season_throws(
    season=SEASON,
    team_id=TEAM_ID,
    max_games=MAX_GAMES,
    delay=0.15,
)

games[["GameID", "AwayTeamID", "HomeTeamID", "AwayScore", "HomeScore", "Status", "StartTimestamp"]]

In [ ]:
possessions, paths = build_scoring_possessions(throws, team_id=TEAM_ID)

print(f"Throws loaded: {len(throws):,}")
print(f"Scoring possessions found for {TEAM_ID}: {len(possessions):,}")

possessions.sort_values("risk_adjusted_aec_per_throw", ascending=False).head(10)

## Average Scoring Path

The average path is progress-normalized. Each scoring possession is resampled to fixed progress checkpoints from possession start to goal, then the checkpoint coordinates are averaged.

In [ ]:
avg_path = average_scoring_path(paths)
avg_path

In [ ]:
fig = plot_average_scoring_path(
    avg_path,
    paths=paths,
    title=f"{TEAM_ID.title()} average scoring path, {SEASON} sample",
    show_individual_paths=True,
)
fig.show()

## Catch Location Heatmap

This shows where completed throws in scoring possessions are caught.

In [ ]:
heatmap = plot_scoring_heatmap(
    paths,
    title=f"{TEAM_ID.title()} scoring-possession catch heatmap, {SEASON} sample",
)
heatmap.show()

## Full Team Season

After the sample plots look right, run the full team season by setting `MAX_GAMES = None` below.

In [ ]:
MAX_GAMES = None

games_full, throws_full = fetch_shownspace_season_throws(
    season=SEASON,
    team_id=TEAM_ID,
    max_games=MAX_GAMES,
    delay=0.15,
)

possessions_full, paths_full = build_scoring_possessions(throws_full, team_id=TEAM_ID)
avg_path_full = average_scoring_path(paths_full)

print(f"Games loaded: {len(games_full):,}")
print(f"Throws loaded: {len(throws_full):,}")
print(f"Scoring possessions found for {TEAM_ID}: {len(possessions_full):,}")

possessions_full.sort_values("risk_adjusted_aec_per_throw", ascending=False).head(20)

In [ ]:
fig_full = plot_average_scoring_path(
    avg_path_full,
    paths=paths_full,
    title=f"{TEAM_ID.title()} average scoring path, {SEASON}",
    show_individual_paths=True,
)
fig_full.show()

In [ ]:
heatmap_full = plot_scoring_heatmap(
    paths_full,
    title=f"{TEAM_ID.title()} scoring-possession catch heatmap, {SEASON}",
)
heatmap_full.show()